In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt 
from data_processing.processing.figure_of_merit import fit_fom, FOM, gaussian, n_sigma_classifier,  bimodal
from data_processing.reporting.plotting import plot_fom, plot_scatter, plot_classification_with_grouping
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from scipy.stats import linregress
from scipy.signal import savgol_filter

In [ ]:
PARQ_ROOT = Path("C:/Users/User/Desktop/FOM Analysis/AmBe for FOM Analysis 2/psd")
# SIG_ROOT = Path("C:/Users/User/Desktop/FOM Analysis/Sig")
# PARQ_PATHS = [path for path in PARQ_ROOT.iterdir()]
# SIG_PATHS = [path for path in SIG_ROOT.iterdir()]

In [ ]:
psd_report = pd.read_parquet(PARQ_ROOT, columns=["CALIB_ENERGY","ENERGYSHORT", "ENERGY", "TIMETAG"])
psd_report["tail / total"] = (psd_report["ENERGY"] - psd_report["ENERGYSHORT"]) / psd_report["ENERGY"]

In [ ]:
fig, ax = plt.subplots(figsize=(12,12))

ax.scatter(
    psd_report["ENERGY"], psd_report["ENERGY"] - psd_report["ENERGYSHORT"], s=0.5, alpha=0.8
)

ax.set_ylim(0,1000)
ax.set_ylabel("Tail")
ax.set_xlabel("Total")

In [ ]:
# psd_report = psd_report.drop(["ENERGY","ENERGYSHORT"], axis=1)
psd_report = psd_report.dropna()
psd_report = psd_report[psd_report["tail / total"].between(0,0.5)]
psd_report.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12,12))

ax.scatter(
    psd_report["CALIB_ENERGY"], psd_report["tail / total"], s=0.5, alpha=0.8
)

In [ ]:
resolution = int(1024/2)

x, y = psd_report["CALIB_ENERGY"], psd_report["tail / total"]
Z, xe, ye = np.histogram2d(x, y, resolution)

In [ ]:
# generating psd/energy 2D histogram
fig, ax = plt.subplots(figsize=(8,8))
cmap = plt.colormaps["nipy_spectral"]
pcm = ax.pcolormesh(xe, ye, Z.T, cmap=cmap)

fontsize = 24
ax.set_xlim(xe[0],1)
ax.set_title("Heatmap of Counts", fontsize=fontsize)
ax.set_ylabel("PSD", fontsize=fontsize)
ax.set_xlabel("Energy (MeVee)", fontsize=fontsize)
ax.tick_params(axis='both', which='major', labelsize=22)
ax.tick_params(axis='both', which='minor', labelsize=22)
fig.colorbar(pcm, ax=ax)

In [ ]:
# slice width
f"Energy Slice Width = {xe[2] - xe[1]:.4f} MeVee"

In [ ]:
# histogram contour plot (vaporwave island)
contour_res = 120
angle_elev = 10
angle_rot = -180

fig = plt.figure(figsize=(12,12))
ax = plt.axes(projection='3d')

x, y = np.meshgrid(xe[:-1], ye[:-1])

ax.view_init(angle_elev, angle_rot)
ax.contour3D(x, y, (Z.T), contour_res, cmap=cmap)
# ax.contour([0.35, 0.35], [0, 0], [100, 100])
ax.set_title("PSD Histogram of Experiments", fontsize=16)
ax.set_ylabel("PSD", fontsize=16)
ax.set_xlabel("Energy (MeVee)", fontsize=16)
ax.set_zlabel("(counts)", fontsize=16)
# ax.vlines(0.35, 0, 100)

In [ ]:
def split_params(params):
    params = abs(params)
    return params[0:3], params[3:]

def log_bimodal(x, mu1, sigma1, a1, mu2, sigma2, a2):
    return np.log(bimodal(x, mu1, sigma1, a1, mu2, sigma2, a2))

def scan_histogram_slices(bin_lbs, histogram, bounds, start_idx, end_idx):
    # end_idx is not inclusive (like end of range())
    end_idx = min(len(histogram), end_idx)
    slice_params = []
    slice_err = []
    for i in range(start_idx, end_idx):
        # get 
#         print(i)
        bounds = (
            (0.01, 0.01, 1, 
             0.25, 0.01, 0),
            (0.23, 0.1, Z.max(),
             0.36, 0.036, 1000)
        )

        
        params, cov = curve_fit(
        bimodal,
        bin_lbs,
        histogram[:,i],
        bounds=bounds
    )
        perr = np.sqrt(np.diag(cov))
        gamma_params, neutron_params = split_params(params)
        fom = FOM(*gamma_params[:-1], *neutron_params[:-1])
        slice_params.append((i, *params, fom))
        slice_err.append((i, *perr))

    df = pd.DataFrame(slice_params, columns=['i', 'mu1', 'sigma1', 'a1', 'mu2', 'sigma2', 'a2', 'fom'])
    err_df = pd.DataFrame(slice_err, columns=['i', 'mu1', 'sigma1', 'a1', 'mu2', 'sigma2', 'a2'])
    return df, err_df

def find_threshold_fom_slice(bin_lbs, histogram, bounds, start_idx, end_idx):
    # end_idx is not inclusive (like end of range())
    # TODO redefine in terms of scan_histogram_slices()
    fom_idxs = []
    fom_params = []
    fom_vals = []
    prev_fom = 0
    
    end_idx = min(len(histogram), end_idx)
    
    for i in range(start_idx, end_idx):       
        params, _ = curve_fit(
            bimodal,
            bin_lbs,
            histogram[:,i],
            bounds=bounds,
        )

        gamma_params, neutron_params = split_params(params)

        fom = FOM(*gamma_params[:-1], *neutron_params[:-1])
        
        if (prev_fom < 1.269) and (round(fom,3) >= 1.27):
            fom_idxs.append(i)
            fom_params.append(params)
            fom_vals.append(fom)
        
        prev_fom = fom
        
    return pd.DataFrame(zip(fom_idxs, fom_vals, fom_params), columns=["index", "fom", "params"])

In [ ]:
psd_bin_lbs = ye[:-1]


bounds = (
    (0.01, 0.01, 1, 
     0.25, 0.02, 0),
    (0.23, 0.2, Z.max(),
     0.36, 0.036, 1000)
)

df_fom = find_threshold_fom_slice(psd_bin_lbs, Z.T, bounds, 0, 240)

In [ ]:
df_fom

In [ ]:
# looking at threshold FOM slice

fom_idx, fom, params = df_fom.iloc[0,:]
fontsize = 22
histogram_slice = Z.T[:,fom_idx]

fig, ax = plt.subplots(figsize=(8,8))
ax.plot(psd_bin_lbs, histogram_slice, lw=3, label="Original")
ax.plot(psd_bin_lbs, bimodal(psd_bin_lbs, *params), '--', lw=3, label="Fitted")
# ax.set_ylim(0, 50)
ax.set_yscale("log")
gamma_params, neutron_params = split_params(params)

ax.set_title(f"FOM={fom:.4f}", fontsize=fontsize)
ax.set_ylabel("Counts", fontsize=fontsize)
ax.set_xlabel("PSD", fontsize=fontsize)
ax.legend(fontsize=fontsize)
ax.tick_params(axis='both', which='major', labelsize=22)
ax.tick_params(axis='both', which='minor', labelsize=22)
ax.grid()

In [ ]:
# scan PSD/energy histogram

# fit_bounds = (
#     (0.01, 0.01, 1, 
#      0.34, 0.02, 0),
#     (0.23, 0.2, Z.max(),
#      0.36, 0.036, 100)
# )

# fit_bounds = (
#     (0.01, 0.01, 1, 
#      0.29, 0.0002, 0),
#     (0.23, 0.2, Z.max(),
#      0.34, 0.036, 100)
# )

start_scan = 0
end_scan = 350
df, err_df = scan_histogram_slices(psd_bin_lbs, Z.T, bounds, start_scan, end_scan)

In [ ]:
# super-awesome histogram scan animation
%matplotlib notebook
from matplotlib.animation import FuncAnimation

fig = plt.figure(figsize=(8,8))
ax = plt.axes(xlim=(0, 0.5), ylim=(0,6e3))
# ax.set_scale("log")
line1, = ax.plot([], [], 'k', lw=3, label='Actual')
line2, = ax.plot([], [], 'r--', lw=3, label='Fit')

def init():
    line1.set_data([], [])
    line2.set_data([], [])
    ax.legend()
    ax.set_xlabel('PSD')
    ax.set_ylabel('N')
    return line1, line2
def animate(i):
    x = psd_bin_lbs
    y = Z.T[:,i]
    line1.set_data(x, y)
    params = tuple(df.iloc[i,1:7])
    fom = df['fom'][i]
    y = bimodal(x, *params)
    line2.set_data(x, y)
#     mu1, _, a1, mu2, _, a2 = params
#     ax.vlines([mu1, mu2], [0, 0], [a1, a2])
    ax.set_title(f"i={i}, E={xe[i]:.3f} MeVee, FOM={fom:.3f}")

    return line1, line2

anim = FuncAnimation(fig, animate, init_func=init, frames=range(0, 10), interval=500, blit=True)

# anim.save('graph.gif', writer='pillow')

In [ ]:
%matplotlib inline

In [ ]:
slice_idx = fom_idx

fig, ax = plt.subplots(figsize=(8,8))
ax.plot(psd_bin_lbs, Z.T[:,slice_idx], "k", lw=4)
params = tuple(df.iloc[slice_idx,1:7])
# params[-2] = params[-2] + 0.01
ax.plot(psd_bin_lbs, bimodal(psd_bin_lbs, *params), "r--", lw=4)
# ax.set_ylim(0, 100)
# ax.vlines(df["mu2"][slice_idx], 0, max(bimodal(psd_bin_lbs, *params)), ls="--", lw=4)
ax.grid()
ax.set_yscale("log")
ax.set_ylabel("Counts", fontsize=fontsize)
ax.set_xlabel("PSD", fontsize=fontsize)
ax.set_title(f"i={slice_idx}; FOM={fom:.3f}", fontsize=fontsize)
ax.tick_params(axis='both', which='major', labelsize=22)
ax.tick_params(axis='both', which='minor', labelsize=22)

In [ ]:
def fit_to_poly(x, y, degree):
    return np.poly1d(np.polyfit(x, y, degree))

def get_param_slice(dataframe, col_label, min, max):
    return dataframe[col_label][min:max]

In [ ]:
# index vs FOM
# fom_fit = fit_to_poly(df['fom'][2:10], xe[2:10], 3)
fontsize = 22
fom_space = np.linspace(1, 1.9, 100)

fig, ax = plt.subplots(figsize=(8,8))

ax.plot(xe[1:end_scan], df['fom'][1:], label="Original", lw=4)
ax.plot(xe[fom_idx], df['fom'][fom_idx],"o")
ax.plot(xe[fom_idx-1], df['fom'][fom_idx-1],"o")
# ax.plot(fom_fit(fom_space), fom_space, '--', label="Fitted", lw=4)
ax.hlines(1.27, 0, 1.25, color='k', linestyle='--', alpha=0.5, lw=4)
# ax.text(*(0.50, 1.27 + 0.01), "FOM=1.27", fontsize=fontsize)

ax.set_ylim(0.8, 1.6)
ax.set_xlim(0.28, 0.35)
ax.set_title("FOM vs Energy Slice", fontsize=fontsize)
ax.set_ylabel("FOM", fontsize=fontsize)
ax.set_xlabel("Energy (MeVee)", fontsize=fontsize)

ax.legend(fontsize=fontsize)
ax.grid()
# ax.set_yticks(np.arange(1,2.25,0.25))
ax.tick_params(axis='both', which='major', labelsize=22)
ax.tick_params(axis='both', which='minor', labelsize=22)

In [ ]:
fom_fit = interp1d(df["fom"][fom_idx-1:fom_idx+1], xe[fom_idx-1:fom_idx+1], kind="linear")


L0 = fom_fit(1.27)
f"FOM ENERGY CUTOFF = {L0:.4f} MeVee"

In [ ]:
all_slice_xs = xe[:end_scan]

In [ ]:
def interp(x, y):
    return interp1d(x, y, "previous", fill_value="extrapolate")

def neutron_ub_fit(x):
    return neutron_lb_fit(x) + window_offset

In [ ]:
# window fitting

window_adj_offset = 0
def classify(psd_report, lb_fit_fn, ub_fit_fn, label):
    neutron_filter = psd_report["tail / total"].between(
        lb_fit_fn(psd_report["CALIB_ENERGY"].astype(float)) + window_adj_offset, ub_fit_fn(psd_report["CALIB_ENERGY"].astype(float))
    ) & (psd_report["CALIB_ENERGY"] >= L0)
    psd_report[label] = neutron_filter
    
    return psd_report

def plot_classification(neutrons, gammas, lb_fit, ub_fit, max_energy, FOM_cutoff, n_neutrons):
    fig, ax = plt.subplots(figsize=(10,10))
    ax.scatter(gammas["CALIB_ENERGY"], gammas["tail / total"], s=2, label="Gammas")
    ax.scatter(neutrons["CALIB_ENERGY"], neutrons["tail / total"], s=2, label="Neutrons")

    energy_space = np.linspace(xe[0], max_energy + 0.5, 200)

    ax.plot(energy_space, lb_fit(energy_space), 'r--')
    ax.plot(energy_space, ub_fit(energy_space), 'r--')
    
    ax.vlines(all_slice_xs[0], lb_fit(all_slice_xs[0]), ub_fit(all_slice_xs[0]), 'r', ls='--')
    # ax.vlines(max_energy, lb_fit(max_energy), ub_fit(max_energy), 'r', ls='--')

    ax.vlines(FOM_cutoff, lb_fit(FOM_cutoff), ub_fit(FOM_cutoff), 'k', alpha=0.5)

    ax.set_title(f"N > FOM ={n_neutrons}", fontsize=fontsize)
    ax.set_ylim(0,0.55)
    ax.set_xlim(0, max_energy + .05)
    
    ax.tick_params(axis='both', which='major', labelsize=22)
    ax.tick_params(axis='both', which='minor', labelsize=22)
    
    ax.set_xlabel("Energy (MeVee)", fontsize=fontsize)
    ax.set_ylabel("PSD", fontsize=fontsize)
    ax.legend(fontsize=fontsize)
    return fig, ax

In [ ]:
sample_frac = 0.01
random_state = 1323

## NASA  


In [ ]:
window_offset = 0.2
sigma = 5
neutron_lb = savgol_filter(df["mu1"] + sigma * df["sigma1"], window_length=21, polyorder=3)
neutron_lb_fit = interp1d(all_slice_xs, neutron_lb, fill_value=(neutron_lb[0], neutron_lb[-1]), bounds_error=False)

plt.plot(all_slice_xs, neutron_lb_fit(all_slice_xs))
plt.plot(all_slice_xs, neutron_ub_fit(all_slice_xs))

In [ ]:
psd_report = classify(psd_report, neutron_lb_fit, neutron_ub_fit, "NASA")

In [ ]:
graph_sample = psd_report.sample(frac=sample_frac,random_state=random_state)

In [ ]:
plot_classification(
    graph_sample[graph_sample["NASA"]], 
    graph_sample[~graph_sample["NASA"]], 
    neutron_lb_fit, 
    neutron_ub_fit, 
    psd_report["CALIB_ENERGY"].max(), 
    L0,
    psd_report[psd_report["NASA"]].shape[0]
)


# plt.plot(all_slice_xs, df["mu2"])
# plt.plot(all_slice_xs, df["mu1"] + 4 * df["sigma1"])
# plt.xlim(0.2,0.4)
# plt.ylim(0.3, 0.5)

## Straight Edge

In [ ]:
slice_start = fom_idx
neutron_straight_lb = df["mu2"][slice_start:] - 3 * df["sigma2"][slice_start:]
neutron_straight_lb_fit = linregress(all_slice_xs[slice_start:], neutron_straight_lb)

In [ ]:
neutron_straight_lb_fit.intercept

In [ ]:
def straight_lb(x):
    return neutron_straight_lb_fit.intercept

def straight_ub(x):
    return 0.5

In [ ]:
psd_report = classify(psd_report, straight_lb, straight_ub, "straight")

In [ ]:
del graph_sample
graph_sample = psd_report.sample(frac=sample_frac,random_state=random_state)

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))
ax.scatter(
    psd_report[~psd_report["straight"]]["CALIB_ENERGY"], 
    psd_report[~psd_report["straight"]]["tail / total"], 
    s=2, 
    label="Gammas"
)
ax.scatter(
    psd_report[psd_report["straight"]]["CALIB_ENERGY"], 
    psd_report[psd_report["straight"]]["tail / total"], 
    s=2, 
    label="Neutrons"
)

ax.hlines(neutron_straight_lb_fit.intercept, 0, psd_report["CALIB_ENERGY"].max() + .05, "k", ls="dashed")
ax.hlines(0.5, 0, psd_report["CALIB_ENERGY"].max() + .05, "k", ls="dashed")

ax.vlines(L0, neutron_straight_lb_fit.intercept, 0.5, 'k', alpha=0.5)

ax.set_title(f"N > FOM ={psd_report[psd_report['straight']].shape[0]}", fontsize=fontsize)
ax.set_ylim(0, 0.55)
ax.set_xlim(0, psd_report["CALIB_ENERGY"].max() + .05)

ax.tick_params(axis='both', which='major', labelsize=22)
ax.tick_params(axis='both', which='minor', labelsize=22)

ax.set_xlabel("Energy (MeVee)", fontsize=fontsize)
ax.set_ylabel("PSD", fontsize=fontsize)
ax.legend(fontsize=fontsize)

# ax.set_xlim(0.29, 0.31)

# CPS

In [ ]:
total_time = psd_report["TIMETAG"].max() * 1e-12
# f"{psd.shape[0] - timedata[timedata['TIMETAG'] < 0].shape[0]:_}"

In [ ]:
total_time

In [ ]:
def counts_over_time_histogram(df, dwell_time, total_time):
    n_bins = int(total_time / dwell_time)   
    print(n_bins)
    counts, bins = np.histogram(df, n_bins)
#     err = np.sqrt(counts / dwell_time)
    return counts[1:], bins[1:]

## NASA Style

In [ ]:
dwell_time = 60 #s

neutron_time_counts, neutron_time_bins = counts_over_time_histogram(
    psd_report[psd_report["NASA"]]["TIMETAG"] * 1e-12, dwell_time, total_time
)

gamma_time_counts, gamma_time_bins = counts_over_time_histogram(
    psd_report[~psd_report["NASA"]]["TIMETAG"]  * 1e-12, dwell_time, total_time
)

In [ ]:
fig, ax = plt.subplots(figsize=(12,8), dpi=120)

ax.plot(neutron_time_bins[:-1], neutron_time_counts, "o--", label="Neutrons Counts")
ax.plot(gamma_time_bins[:-1], gamma_time_counts, "o--", label="Gamma Counts")
ax.legend()
ax.set_yscale("log")
# ax.set_ylim(1, 0.25e6)
ax.set_title("Event Counts per Dwell Time over Experiment Time")
ax.set_ylabel("$\log$(Event Counts)", fontsize=14)
ax.set_xlabel("Time (s)", fontsize=14)
# ax.grid()

In [ ]:
neutron_time_cps = neutron_time_counts / dwell_time
gamma_time_cps = gamma_time_counts / dwell_time

In [ ]:
fig, ax = plt.subplots(figsize=(12,8), dpi=120)

ax.plot(neutron_time_bins[:-1], neutron_time_cps, "o--", label="Neutrons CPS")
ax.plot(gamma_time_bins[:-1], gamma_time_cps, "o--", label="Gamma CPS")
ax.legend()
# ax.set_yscale("log")
# ax.set_ylim(2, 3.5)
ax.set_title("Count Rate over Experiment Time")
ax.set_ylabel("Count Rate", fontsize=14)
ax.set_xlabel("Time (s)", fontsize=14)

In [ ]:
fig, ax = plt.subplots(figsize=(12,8), dpi=120)

ax.plot(gamma_time_bins[:-1], gamma_time_cps, "o--", c="orange", label="Gamma CPS")
ax.legend()
# ax.set_yscale("log")
# ax.set_ylim(1500, 2000)
ax.set_title("Gamma Count Rate over Experiment Time")
ax.set_ylabel("Count Rate", fontsize=14)
ax.set_xlabel("Time (s)", fontsize=14)

In [ ]:
cps_n_bins = 200

neutron_cps_counts, neutron_cps_bins = np.histogram(neutron_time_cps, cps_n_bins)
gamma_cps_counts, gamma_cps_bins = np.histogram(gamma_time_cps, cps_n_bins)

In [ ]:
gamma_cps_bins[2] - gamma_cps_bins[1]

In [ ]:
fig, ax = plt.subplots(figsize=(8,8), dpi=120)

ax.stairs(neutron_cps_counts, neutron_cps_bins, label="Neutrons")
# ax.plot(neutron_cps_bins[:-1], neutron_cps_counts, label="Neutrons")
ax.stairs(gamma_cps_counts, gamma_cps_bins, label="Gamma", color="orange")
# ax.set_ylim(-, 500)
ax.set_xlim(500,600)
# ax.set_yscale("log")
# ax.set_xscale("log")
ax.legend()
ax.grid()
ax.set_xlabel("CPS")
ax.set_ylabel("Counts")
ax.set_title(f"Gamma CPS with dwell Time ={dwell_time}s")

## Straight Edge

In [ ]:
dwell_time = 10 #s

neutron_time_counts, neutron_time_bins = counts_over_time_histogram(
    psd_report[psd_report["straight"]]["TIMETAG"] * 1e-12, dwell_time, total_time
)

gamma_time_counts, gamma_time_bins = counts_over_time_histogram(
    psd_report[~psd_report["straight"]]["TIMETAG"]  * 1e-12, dwell_time, total_time
)

In [ ]:
fig, ax = plt.subplots(figsize=(12,8), dpi=120)

ax.plot(neutron_time_bins[:-1], neutron_time_counts, "o--", label="Neutrons Counts")
ax.plot(gamma_time_bins[:-1], gamma_time_counts, "o--", label="Gamma Counts")
ax.legend()
# ax.set_yscale("log")
# ax.set_ylim(1, 0.25e6)
ax.set_title("Event Counts per Dwell Time over Experiment Time")
ax.set_ylabel("$\log$(Event Counts)", fontsize=14)
ax.set_xlabel("Time (s)", fontsize=14)
# ax.grid()

In [ ]:
neutron_time_cps = neutron_time_counts / dwell_time
gamma_time_cps = gamma_time_counts / dwell_time

In [ ]:
fig, ax = plt.subplots(figsize=(12,8), dpi=120)

ax.plot(neutron_time_bins[:-1], neutron_time_cps, "o--", label="Neutrons CPS")
ax.plot(gamma_time_bins[:-1], gamma_time_cps, "o--", label="Gamma CPS")
ax.legend()
# ax.set_yscale("log")
# ax.set_ylim(1, 0.25e6)
ax.set_title("Count Rate over Experiment Time")
ax.set_ylabel("Count Rate", fontsize=14)
ax.set_xlabel("Time (s)", fontsize=14)

In [ ]:
cps_n_bins = 200

neutron_cps_counts, neutron_cps_bins = np.histogram(neutron_time_cps, cps_n_bins)
gamma_cps_counts, gamma_cps_bins = np.histogram(gamma_time_cps, cps_n_bins)

In [ ]:
fig, ax = plt.subplots(figsize=(8,8), dpi=120)

ax.stairs(neutron_cps_counts, neutron_cps_bins, label="Neutrons")
# ax.plot(neutron_cps_bins[:-1], neutron_cps_counts, label="Neutrons")
ax.stairs(gamma_cps_counts, gamma_cps_bins, label="Gamma", color="orange")
# ax.set_ylim(-, 500)
ax.set_xlim(100,2600)
# ax.set_yscale("log")
# ax.set_xscale("log")
ax.legend()
ax.grid()
ax.set_xlabel("CPS")
ax.set_ylabel("Counts")
ax.set_title(f"Neutron CPS with dwell Time ={dwell_time}s")